# Downtown Toronto Hospitals Data Collection

This notebook contains the code to scrape:
1. **Qualitative Data (News)**: Using Google News RSS feeds to find articles related to downtown Toronto hospitals from the last 5 years.
2. **Quantitative Data (Open Data)**: Using the `opendatatoronto` library to pull healthcare institutional data, and highlighting how to analyze metrics from the last 5 years.

In [27]:
import pip
def install(package):
    if hasattr(pip, 'main'):
        pip.main(['install', package])
    else:
        pip._internal.main(['install', package])

# Install necessary packages if they don't exist
try:
    import feedparser
    import requests
except ImportError: 
    install('feedparser')
    install('requests')
    import feedparser
    import requests


## 1. Qualitative Data: Scraping News (Last 5 Years)
We use the Google News RSS feed, which allows searching by query, and supports the `after:` and `before:` date operators.

In [28]:
import urllib.parse
import feedparser
import pandas as pd
from datetime import datetime, timedelta
import time

# List of major downtown Toronto hospitals
downtown_hospitals = [
    "Toronto General Hospital",
    "Mount Sinai Hospital Toronto",
    "St. Michael's Hospital Toronto",
    "SickKids Hospital Toronto",
    "Toronto Western Hospital",
    "Princess Margaret Cancer Centre",
    "Women's College Hospital"
]

def _format_df(df):
    """Standardize dataframe format: date first, sorted columns, no scraping indicator columns."""
    if df.empty:
        return pd.DataFrame(columns=["date"])

    date_candidates = [c for c in df.columns if "date" in c.lower()]
    if "date" not in df.columns and date_candidates:
        df = df.rename(columns={date_candidates[0]: "date"})

    if "date" not in df.columns:
        df["date"] = pd.NaT

    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    # Remove scraping-indicator fields if present
    drop_cols = [c for c in df.columns if c.lower() in {"keyword", "category"}]
    if drop_cols:
        df = df.drop(columns=drop_cols)

    other_cols = sorted([c for c in df.columns if c != "date"])
    return df[["date"] + other_cols].sort_values(by="date", ascending=False).reset_index(drop=True)

def scrape_google_news(query, start_date, end_date):
    """
    Scrapes Google News RSS for a given query and date range.
    Dates should be in YYYY-MM-DD format.
    """
    search_query = f'"{query}" after:{start_date} before:{end_date}'
    url_encoded_query = urllib.parse.quote(search_query)
    rss_url = f"https://news.google.com/rss/search?q={url_encoded_query}&hl=en-CA&gl=CA&ceid=CA:en"

    feed = feedparser.parse(rss_url)
    articles = []

    for entry in feed.entries:
        articles.append({
            "hospital": query,
            "title": entry.title,
            "source": entry.source.title if hasattr(entry, "source") else "Unknown",
            "link": entry.link,
            "published_date": entry.published,
        })

    return articles

# Define the last 5 years date range
end_date = datetime.now()
start_date = end_date - timedelta(days=5 * 365)

start_date_str = start_date.strftime("%Y-%m-%d")
end_date_str = end_date.strftime("%Y-%m-%d")

all_qualitative_data = []
for hospital in downtown_hospitals:
    all_qualitative_data.extend(scrape_google_news(hospital, start_date_str, end_date_str))
    time.sleep(1.5)

news_df = pd.DataFrame(all_qualitative_data)
news_df = _format_df(news_df)
news_df


,date,hospital,link,source,title
0,2026-03-13 15:11:56,Princess Margaret Cancer Centre,https://news.google.com/rss/articles/CBMiW0FVX...,Oncodaily,Nihar Desai: A Remarkable Year at Princess Mar...
1,2026-03-10 21:51:13,Toronto General Hospital,https://news.google.com/rss/articles/CBMia0FVX...,UHN Foundation,"For the first time, UHN’s Toronto General Hosp..."
2,2026-02-26 08:00:00,Toronto Western Hospital,https://news.google.com/rss/articles/CBMikAFBV...,Global News,Toronto’s UHN ranked 2nd best Hospital in the ...
3,2026-02-26 08:00:00,Toronto General Hospital,https://news.google.com/rss/articles/CBMikAFBV...,Global News,Toronto’s UHN ranked 2nd best Hospital in the ...
4,2026-02-25 08:00:00,Toronto General Hospital,https://news.google.com/rss/articles/CBMidEFVX...,UHN Foundation,Proud to be named #2 in the world - UHN Founda...
...,...,...,...,...,...
402,2021-04-16 07:00:00,Toronto Western Hospital,https://news.google.com/rss/articles/CBMi-gFBV...,canada.ca,Noventa Energy Partners using innovative techn...
403,2021-03-24 07:00:00,Toronto Western Hospital,https://news.google.com/rss/articles/CBMijAFBV...,CBC,Thousands of Toronto hospital staff haven't go...
404,2021-03-23 07:00:00,Toronto Western Hospital,https://news.google.com/rss/articles/CBMiZ0FVX...,UHN Foundation,UHN Foundation is here - UHN Foundation
405,2021-03-18 07:00:00,Toronto General Hospital,https://news.google.com/rss/articles/CBMi3AFBV...,Toronto Star,Outbreak declared in Toronto General Hospital ...


## 2. Quantitative Data: Toronto Open Data & Other Sources
We can query the open data portal for Toronto to find datasets related to healthcare institutions and outbreaks (a common quantifiable metric for hospitals).

In [36]:
import pandas as pd
import requests


def _format_df(df):
    """Standardize dataframe format: date first, sorted columns, no scraping indicator columns."""
    if df.empty:
        return pd.DataFrame(columns=["date"])

    date_candidates = [c for c in df.columns if "date" in c.lower()]
    if "date" not in df.columns and date_candidates:
        df = df.rename(columns={date_candidates[0]: "date"})

    if "date" not in df.columns:
        df["date"] = pd.NaT

    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    drop_cols = [c for c in df.columns if c.lower() in {"keyword", "category", "_id"}]
    if drop_cols:
        df = df.drop(columns=drop_cols)

    other_cols = sorted([c for c in df.columns if c != "date"])
    return df[["date"] + other_cols].sort_values(by="date", ascending=False).reset_index(drop=True)


base_url = "https://ckan0.cf.opendata.inter.prod-toronto.ca"
search_url = f"{base_url}/api/3/action/package_search"
params = {"q": "healthcare"}
package_search_res = requests.get(search_url, params=params).json()

healthcare_packages = pd.DataFrame(package_search_res["result"]["results"])

outbreak_pkg_id = "outbreaks-in-toronto-healthcare-institutions"
outbreaks_df = pd.DataFrame()

try:
    package_url = f"{base_url}/api/3/action/package_show"
    package = requests.get(package_url, params={"id": outbreak_pkg_id}).json()

    datastore_resources = [r for r in package["result"]["resources"] if r.get("datastore_active")]

    if datastore_resources:
        data_resource = datastore_resources[0]
        data_url = f"{base_url}/api/3/action/datastore_search"
        data_params = {"id": data_resource["id"], "limit": 100000}
        data_res = requests.get(data_url, params=data_params).json()
        raw_df = pd.DataFrame(data_res["result"]["records"])

        date_cols = [c for c in raw_df.columns if "date" in c.lower()]
        if date_cols:
            main_date_col = date_cols[0]
            raw_df[main_date_col] = pd.to_datetime(raw_df[main_date_col], errors="coerce")
            raw_df = raw_df.rename(columns={main_date_col: "date"})

            try:
                five_years_ago_ts = pd.Timestamp(start_date)
                if raw_df["date"].dt.tz is not None:
                    five_years_ago_ts = five_years_ago_ts.tz_localize(raw_df["date"].dt.tz)
                raw_df = raw_df[raw_df["date"] >= five_years_ago_ts]
            except NameError:
                pass
        else:
            raw_df["date"] = pd.NaT

        outbreaks_df = _format_df(raw_df)
except Exception:
    outbreaks_df = pd.DataFrame(columns=["date"])

outbreaks_df


,date,Active,Causative Agent-1,Causative Agent-2,Date Declared Over,Institution Address,Institution Name,Outbreak Setting,Type of Outbreak
0,2026-03-11,Y,Pending,None,None,5935 Bathurst St,Cheltenham Care Community - 2nd Fl South,LTCH,Respiratory
1,2026-03-11,Y,Pending,None,None,767 Royal York Rd,Ivan Franko Home - 1st Fl,LTCH,Respiratory
2,2026-03-10,Y,Respiratory syncytial virus,None,None,1 Northwestern Ave,Harold and Grace Baker Centre - Wing A,LTCH,Respiratory
3,2026-03-09,Y,Pending,None,None,1000 Ellesmere Rd,Fieldstone Commons Care Community - 3A,LTCH,Respiratory
4,2026-03-09,Y,Pending,None,None,70 Humberline Dr,Deerwood Creek Community - 3rd Fl - Ivy Wood,LTCH,Respiratory
...,...,...,...,...,...,...,...,...,...
210,2026-01-02,N,COVID-19,None,2026-01-17,205 Cummer Ave,Cummer Lodge - 3 South,LTCH,Respiratory
211,2026-01-02,N,Influenza A (Not subtyped),None,2026-01-18,102 Craiglee Dr,Craiglee Nursing Home - 2E,LTCH,Respiratory
212,2026-01-02,N,Influenza A (Not subtyped),None,2026-01-13,66 Roncesvalles Ave,Copernicus Lodge - 4th Fl South,LTCH,Respiratory
213,2026-01-01,N,Influenza A (Not subtyped),None,2026-01-15,70 Humberline Dr,Deerwood Creek Community - 2B & 3B,LTCH,Respiratory


## 3. Web Scraping Alternative (Optional)
If APIs are insufficient, we can scrape qualitative text from hospital pages (news, annual reports, quality reports) and extract key signals related to hospital quality. This version captures snippets tied to funding, wait times, staffing, patient safety, and access/equity.

In [30]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime, timezone

QUALITY_KEYWORDS = {
    "funding": [
        "funding", "investment", "budget", "capital", "grant", "donation", "foundation"
    ],
    "wait_time": [
        "wait time", "waiting time", "er wait", "emergency wait", "surgical backlog", "queue"
    ],
    "staffing": [
        "staffing", "nurse", "physician", "hiring", "vacancy", "retention", "workforce"
    ],
    "quality_safety": [
        "patient safety", "infection", "readmission", "mortality", "quality improvement", "accreditation"
    ],
    "access_equity": [
        "equity", "access", "underserved", "indigenous", "language access", "mental health access"
    ],
}

def _format_df(df):
    """Standardize dataframe format: date first, sorted columns, no scraping indicator columns."""
    if df.empty:
        return pd.DataFrame(columns=["date"])

    date_candidates = [c for c in df.columns if "date" in c.lower()]
    if "date" not in df.columns and date_candidates:
        df = df.rename(columns={date_candidates[0]: "date"})

    if "date" not in df.columns:
        df["date"] = pd.NaT

    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    drop_cols = [c for c in df.columns if c.lower() in {"keyword", "category"}]
    if drop_cols:
        df = df.drop(columns=drop_cols)

    other_cols = sorted([c for c in df.columns if c != "date"])
    return df[["date"] + other_cols].sort_values(by="date", ascending=False).reset_index(drop=True)

def _clean_text(text):
    return re.sub(r"\s+", " ", text).strip()

def scrape_hospital_quality_signals(url, keyword_map=QUALITY_KEYWORDS):
    """
    Scrape a hospital webpage and return snippet-level qualitative rows.
    Output intentionally excludes scraping metadata columns (keyword/category).
    """
    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")
        text_nodes = soup.find_all(["h1", "h2", "h3", "h4", "p", "li"])

        snippets = []
        seen = set()
        for node in text_nodes:
            txt = _clean_text(node.get_text(" ", strip=True))
            if len(txt) < 40 or txt in seen:
                continue
            seen.add(txt)
            snippets.append(txt)

        collected_date = pd.Timestamp(datetime.now(timezone.utc).date())
        rows = []
        for snippet in snippets:
            lower_snippet = snippet.lower()
            matched = any(
                word in lower_snippet
                for words in keyword_map.values()
                for word in words
            )
            if matched:
                rows.append({
                    "date": collected_date,
                    "source_url": url,
                    "snippet": snippet,
                })

        return pd.DataFrame(rows, columns=["date", "source_url", "snippet"])
    except Exception as e:
        print(f"Failed to scrape {url}: {e}")
        return pd.DataFrame(columns=["date", "source_url", "snippet"])

def scrape_multiple_hospital_pages(urls):
    frames = [scrape_hospital_quality_signals(url) for url in urls]
    frames = [f for f in frames if not f.empty]
    if not frames:
        return pd.DataFrame(columns=["date", "source_url", "snippet"])

    qualitative_df = pd.concat(frames, ignore_index=True).drop_duplicates().reset_index(drop=True)
    return _format_df(qualitative_df)

hospital_pages = [
    "https://www.uhn.ca/corporate/News",
    "https://sunnybrook.ca/content/?page=about-us-newsroom",
]

qualitative_df = scrape_multiple_hospital_pages(hospital_pages)

qualitative_df.shape


Failed to scrape https://sunnybrook.ca/content/?page=about-us-newsroom: 404 Client Error: Not Found for url: https://sunnybrook.ca/content/?page=about-us-newsroom


(5, 3)

In [35]:
# 4. Compile all dataframes and sort by date
import pandas as pd


def _prepare_for_merge(df, name):
    print(f"Processing {name}...")

    if df is None or df.empty:
        print(f"- {name}: 0 articles")
        return pd.DataFrame(columns=["date", "dataset"])

    merged_df = df.copy()

    if "date" not in merged_df.columns:
        date_candidates = [c for c in merged_df.columns if "date" in c.lower()]
        if date_candidates:
            merged_df = merged_df.rename(columns={date_candidates[0]: "date"})
        else:
            merged_df["date"] = pd.NaT

    merged_df["date"] = pd.to_datetime(merged_df["date"], errors="coerce")

    # Remove scraping-indicator columns if present
    drop_cols = [c for c in merged_df.columns if c.lower() in {"keyword", "category"}]
    if drop_cols:
        merged_df = merged_df.drop(columns=drop_cols)

    merged_df["dataset"] = name
    print(f"- {name}: {len(merged_df)} articles")
    return merged_df

frames = [
    _prepare_for_merge(news_df, "news_df"),
    _prepare_for_merge(outbreaks_df, "outbreaks_df"),
    _prepare_for_merge(qualitative_df, "qualitative_df"),
]

compiled_df = pd.concat(frames, ignore_index=True, sort=False)
compiled_df = compiled_df.sort_values(by="date", ascending=False).reset_index(drop=True)

ordered_cols = ["date"] + sorted([c for c in compiled_df.columns if c != "date"])
compiled_df = compiled_df[ordered_cols]

print(f"\nTotal compiled records: {len(compiled_df)}")
print(compiled_df)
compiled_df

Processing news_df...
- news_df: 407 articles
Processing outbreaks_df...
- outbreaks_df: 215 articles
Processing qualitative_df...
- qualitative_df: 5 articles

Total compiled records: 627
                   date Active Causative Agent-1 Causative Agent-2  \
0   2026-03-15 00:00:00    NaN               NaN               NaN   
1   2026-03-15 00:00:00    NaN               NaN               NaN   
2   2026-03-15 00:00:00    NaN               NaN               NaN   
3   2026-03-15 00:00:00    NaN               NaN               NaN   
4   2026-03-15 00:00:00    NaN               NaN               NaN   
..                  ...    ...               ...               ...   
622 2021-04-16 07:00:00    NaN               NaN               NaN   
623 2021-03-24 07:00:00    NaN               NaN               NaN   
624 2021-03-23 07:00:00    NaN               NaN               NaN   
625 2021-03-18 07:00:00    NaN               NaN               NaN   
626 2021-03-18 07:00:00    NaN           

,date,Active,Causative Agent-1,Causative Agent-2,Date Declared Over,Institution Address,Institution Name,Outbreak Setting,Type of Outbreak,dataset,hospital,link,snippet,source,source_url,title
0,2026-03-15 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,qualitative_df,NaN,NaN,About UHN About UHN University Health Network ...,NaN,https://www.uhn.ca/corporate/News,NaN
1,2026-03-15 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,qualitative_df,NaN,NaN,Patients & Visitors Patients & Visitors At UHN...,NaN,https://www.uhn.ca/corporate/News,NaN
2,2026-03-15 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,qualitative_df,NaN,NaN,Our UHN programs and services are among the mo...,NaN,https://www.uhn.ca/corporate/News,NaN
3,2026-03-15 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,qualitative_df,NaN,NaN,Programs & Services Our UHN programs and servi...,NaN,https://www.uhn.ca/corporate/News,NaN
4,2026-03-15 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,qualitative_df,NaN,NaN,Programs & Services Programs & Services Our UH...,NaN,https://www.uhn.ca/corporate/News,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
622,2021-04-16 07:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,news_df,Toronto Western Hospital,https://news.google.com/rss/articles/CBMinAFBV...,NaN,Global News,NaN,2 Toronto hospitals install tents outside to r...
623,2021-03-24 07:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,news_df,Toronto Western Hospital,https://news.google.com/rss/articles/CBMijAFBV...,NaN,CBC,NaN,Thousands of Toronto hospital staff haven't go...
624,2021-03-23 07:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,news_df,Toronto Western Hospital,https://news.google.com/rss/articles/CBMiZ0FVX...,NaN,UHN Foundation,NaN,UHN Foundation is here - UHN Foundation
625,2021-03-18 07:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,news_df,Toronto General Hospital,https://news.google.com/rss/articles/CBMi3AFBV...,NaN,Toronto Star,NaN,Outbreak declared in Toronto General Hospital ...
